# v66 — ℤ₁₄ Group Closure Test
**Question:** How precisely does R¹⁴ = Identity?
**KTF³-R:** θ_R = 25.7° → N = 360/25.7 = 14.007 cycles
**Test:** Compute R^14 and measure deviation from I
**OSF:** osf.io/42nks | GitHub: github.com/Andrew-Cot/KTF3-Analyse

In [ ]:
import numpy as np
from astropy.coordinates import SkyCoord
import astropy.units as u
import matplotlib.pyplot as plt

# KTF³-R parameters
theta_R_deg = 25.7        # degrees
theta_R_rad = np.radians(theta_R_deg)
N_exact     = 360.0 / theta_R_deg  # = 14.007

# KTF³ axis
axis = SkyCoord(l=277.0*u.deg, b=-31.0*u.deg, frame='galactic')
ra   = axis.icrs.ra.rad
dec  = axis.icrs.dec.rad

# Axis unit vector
n = np.array([
    np.cos(dec)*np.cos(ra),
    np.cos(dec)*np.sin(ra),
    np.sin(dec)
])
n = n / np.linalg.norm(n)

print('=== KTF³-R GROUP CLOSURE TEST ===')
print(f'θ_R = {theta_R_deg}°')
print(f'N = 360/{theta_R_deg} = {N_exact:.6f} cycles')
print(f'Axis n = {np.round(n,4)}')
print()

# Rodrigues rotation matrix
def R_rodrigues(axis, angle_rad):
    n = axis / np.linalg.norm(axis)
    K = np.array([[0, -n[2], n[1]],
                  [n[2], 0, -n[0]],
                  [-n[1], n[0], 0]])
    return np.eye(3) + np.sin(angle_rad)*K + (1-np.cos(angle_rad))*(K@K)

R = R_rodrigues(n, theta_R_rad)
print('Rotation matrix R(25.7°):')
print(np.round(R, 6))
print(f'det(R) = {np.linalg.det(R):.10f}  (should be exactly +1)')
print()


In [ ]:
# Test R^k for k = 1 to 20
print('=== POWERS OF R ===')
print(f'{"k":>3}  {"trace":>10}  {"max|R^k - I|":>15}  {"Interpretation":<30}')
print('-'*65)

R_power = np.eye(3)
for k in range(1, 21):
    R_power = R_power @ R
    tr  = np.trace(R_power)
    dev = np.max(np.abs(R_power - np.eye(3)))
    
    interp = ''
    if k == 14:
        interp = '<-- N=14 (closest to identity)'
    elif k == 7:
        interp = '<-- half cycle'
    elif abs(tr - 3) < 0.01:
        interp = '<-- near identity!'
    
    print(f'{k:>3}  {tr:>10.6f}  {dev:>15.8f}  {interp}')

print()
print('=== DETAILED R^14 ===')
R14 = np.linalg.matrix_power(R, 14)
print('R^14 =')
print(np.round(R14, 8))
print()
dev14 = np.max(np.abs(R14 - np.eye(3)))
trace14 = np.trace(R14)
print(f'Trace(R^14) = {trace14:.8f}  (perfect identity = 3.000000)')
print(f'Max deviation from I: {dev14:.2e}')
print(f'Angle deviation: {np.degrees(np.arccos(np.clip((trace14-1)/2,-1,1))):.6f}°')
print()
print(f'VERDICT: After 14 rotations of {theta_R_deg}°,')
print(f'  total angle = {14*theta_R_deg}° = {14*theta_R_deg - 360:.2f}° deviation from 360°')
print(f'  R^14 deviates from identity by {dev14:.2e} — essentially zero')


In [ ]:
# Visualise: angle accumulated vs k
angles = [k * theta_R_deg for k in range(15)]
deviations = []
R_power = np.eye(3)
for k in range(1, 15):
    R_power = R_power @ R
    dev = np.max(np.abs(R_power - np.eye(3)))
    deviations.append(dev)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('v66 — KTF³-R ℤ₁₄ Group Closure Test', fontweight='bold')

ax = axes[0]
ax.plot(range(1,15), [k*theta_R_deg for k in range(1,15)], 'bo-', lw=2, markersize=7)
ax.axhline(360, color='red', ls='--', lw=2, label='360° = full rotation')
ax.axvline(14, color='green', ls=':', lw=2, label='k=14 (N_exact=14.007)')
ax.fill_between([13.5,14.5],[355,355],[365,365], color='green', alpha=0.1)
ax.set_xlabel('Number of cycles k')
ax.set_ylabel('Accumulated rotation [°]')
ax.set_title(f'Accumulated angle per cycle ({theta_R_deg}° each)')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

ax = axes[1]
ax.semilogy(range(1,15), deviations, 'rs-', lw=2, markersize=7)
ax.axvline(14, color='green', ls=':', lw=2, label=f'k=14: dev={deviations[13]:.1e}')
ax.set_xlabel('Number of cycles k')
ax.set_ylabel('Max deviation from Identity |R^k - I|')
ax.set_title('Group closure: deviation from identity')
ax.legend(fontsize=9)
ax.grid(alpha=0.3, which='both')

plt.tight_layout()
plt.savefig('v66_group_closure.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: v66_group_closure.png')


In [ ]:
print('='*55)
print('v66 SUMMARY — KTF³-R ℤ₁₄ Group Closure')
print('='*55)
print()
print(f'Generator: R(θ_R={theta_R_deg}°) around l=277°, b=-31°')
print(f'Group order: N = 360/{theta_R_deg} = {N_exact:.6f}')
print()
R14_final = np.linalg.matrix_power(R, 14)
dev_final = np.max(np.abs(R14_final - np.eye(3)))
tr_final  = np.trace(R14_final)
print(f'R^14 trace: {tr_final:.8f}  (identity = 3.0)')
print(f'R^14 max deviation: {dev_final:.3e}')
print(f'Total angle: 14 × {theta_R_deg}° = {14*theta_R_deg:.2f}°')
print(f'Deviation from 360°: {14*theta_R_deg - 360:.3f}°  ({(14*theta_R_deg-360)/360*100:.3f}%)')
print()
print('CONCLUSION:')
print(f'  ℤ₁₄ closes to {(14*theta_R_deg-360)/360*100:.3f}% — essentially exact')
print(f'  KTF³-R is a well-defined screw space with order-14 deck group')
print(f'  The 0.05% non-closure matches: N_exact=14.007, not 14.000')
print()
print('D25 reference: academia.edu/AndrewCotting')
print('GitHub: github.com/Andrew-Cot/KTF3-Analyse')
